In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('training_runs.csv')

# paper reference lines
PAPER_R1 = 56.5
PAPER_MIOU = 59.2

clip = df[df['feature'] == 'CLIP'].copy()
ib   = df[df['feature'] == 'ImageBind'].copy()

# shift epoch numbers: actual epoch = stored epoch + 1
clip['epoch_plot'] = clip['epoch'] + 1
ib['epoch_plot']   = ib['epoch'] + 1

In [ ]:
import numpy as np
from scipy.interpolate import make_interp_spline
from pathlib import Path

Path('plots').mkdir(exist_ok=True)

def smooth_plot(ax, x, y, color, label, marker):
    x = np.array(x); y = np.array(y)
    ax.plot(x, y, marker=marker, color=color, alpha=0.25, linewidth=1)
    if len(x) >= 4:
        x_smooth = np.linspace(x.min(), x.max(), 300)
        spl = make_interp_spline(x, y, k=3)
        ax.plot(x_smooth, spl(x_smooth), color=color, linewidth=2, label=label)
        ax.plot(x, y, marker=marker, color=color, markersize=6, linestyle='none')
    else:
        ax.plot(x, y, marker=marker, color=color, linewidth=2, label=label)

metrics = [
    ('R1@0.5', 'R@1 IoU=0.5 (%)', PAPER_R1),
    ('R1@0.7', 'R@1 IoU=0.7 (%)', None),
    ('mAP',    'mAP (%)',          None),
    ('mIoU',   'mIoU (%)',         PAPER_MIOU),
    ('loss_f', 'Loss F',           None),
    ('loss_g', 'Loss G',           None),
]

for col, ylabel, paper_val in metrics:
    fig, ax = plt.subplots(figsize=(7, 5))
    smooth_plot(ax, clip['epoch_plot'], clip[col], 'steelblue',  'CLIP',      'o')
    smooth_plot(ax, ib['epoch_plot'],   ib[col],   'darkorange', 'ImageBind', 's')
    if paper_val is not None:
        ax.axhline(paper_val, color='gray', linestyle='--', linewidth=1.2, label=f'Paper MATR ({paper_val})')
    ax.set_title(ylabel, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_xticks(clip['epoch_plot'])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    fname = f"plots/{col.replace('@', '_at_').replace('/', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')

In [ ]:
# Best epoch per run
print('=== Best Epoch by R@1@0.5 ===')
for feat, grp in df[df['epoch'] >= 0].groupby('feature'):
    best = grp.loc[grp['R1@0.5'].idxmax()]
    print(f"{feat:12s}  epoch={int(best['epoch'])}  R1@0.5={best['R1@0.5']:.2f}  R1@0.7={best['R1@0.7']:.2f}  mAP={best['mAP']:.2f}  mIoU={best['mIoU']:.2f}")

print(f"\nPaper MATR (Finetuned+V)  R1@0.5=56.50  mIoU=59.20")